# Whisper Audio Drive — Multi Speaker / Diarization

Notebook ini adalah versi **multi-speaker**. Versi ini sudah diperbaiki untuk konflik dependency Colab terbaru.

Flow:

1. Install WhisperX + pyannote
2. Mount Google Drive
3. Load model transcribe **Whisper large-v3**
4. Load model diarization speaker
5. Cek audio/video di folder Drive
6. Transcribe + deteksi speaker
7. Export ke:
   - `.txt` format timestamp + `[Speaker 0]`, `[Speaker 1]`, dst
   - `.md`
   - `.srt`
   - `.vtt`
   - `.json`

Folder input default:

```txt
/content/drive/MyDrive/Whisper/AudioFiles
```

Folder output default:

```txt
/content/drive/MyDrive/Whisper/Transcripts_MultiSpeaker
```

## Penting

Untuk multi-speaker, kamu butuh **HuggingFace token** karena diarization memakai pyannote.

Langkah singkat:

1. Buat token di HuggingFace: **Settings → Access Tokens**
2. Accept model terms untuk pyannote:
- https://huggingface.co/pyannote/speaker-diarization-3.1
- https://huggingface.co/pyannote/segmentation-3.0
- https://huggingface.co/pyannote/speaker-diarization-community-1

3. Saat notebook minta token, paste token kamu


Aktifkan GPU di Colab:

```txt
Runtime → Change runtime type → T4 GPU / L4 GPU / A100
```


In [3]:
# ============================================================
# CELL 1 — INSTALL DEPENDENCIES, FIXED FOR COLAB
# ============================================================

!apt-get -qq update || true
!apt-get -qq install -y ffmpeg

# Uninstall to avoid conflicts
!pip -q uninstall -y whisperx pyannote.audio pyannote.core pyannote.metrics huggingface_hub pandas numpy

# Install specific compatible versions
# NumPy 1.26.4 is the last stable 1.x version to avoid '_center' import errors
!pip -q install "numpy==1.26.4" "pandas==2.2.2" "huggingface-hub>=0.34.0,<1.0"
!pip -q install git+https://github.com/m-bain/whisperx.git

# Final enforcement to override any sub-dependency upgrades
!pip -q install --upgrade --force-reinstall --no-deps "numpy==1.26.4" "pandas==2.2.2"

print("✅ Dependencies installed.")
print("IMPORTANT: Please go to Runtime -> Restart session now, then run from CELL 2.")

import numpy, pandas, huggingface_hub
print(f"Verified Versions: NumPy {numpy.__version__}, Pandas {pandas.__version__}")

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
pyannote-database 6.1.1 requires pyannote-core>=6.0.0, which is not installed.
pyannote-pipeline 4.0.0 requires pyannote-core>=6.0.0, which is not installed.
pyannote-database 6.1.1 requires pandas>=2.2.3, but you have pandas 2.2.2 which is incompatible.
xarray-einstats 0.10.0 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
jax 0.7.2 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
cupy-cuda12x 14.0.1 requires numpy<2.6,>=2.0, but you have numpy 1.26.4 which is incompatible.
shap 0.52.0 requires numpy>=2, but you have numpy 1.26.4 which is incompatible.
tobler 0.14.0 requires numpy>=2.0

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [4]:
# ============================================================
# CELL 2 — MOUNT GOOGLE DRIVE
# ============================================================

from google.colab import drive

drive.mount("/content/drive")
print("✅ Google Drive mounted.")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ Google Drive mounted.


In [18]:
# ============================================================
# CELL 3 — CONFIG (UPDATED TO FIX OOM)
# ============================================================

import os
import re
import json
import subprocess
from pathlib import Path
from collections import Counter

import torch
from huggingface_hub import login

# Force clear GPU memory from previous failed runs
if torch.cuda.is_available():
    torch.cuda.empty_cache()

# =========================
# FOLDER SETTING
# =========================

AUDIO_FOLDER = "/content/drive/MyDrive/Whisper/AudioFiles"
OUTPUT_FOLDER = "/content/drive/MyDrive/Whisper/Transcripts_MultiSpeaker"

# =========================
# MODEL SETTING
# =========================
MODEL_NAME = "large-v3"
LANGUAGE = "id"

# Reduced BATCH_SIZE to avoid 'Out of Memory' on T4 GPU
BATCH_SIZE = 4

MIN_SPEAKERS = None
MAX_SPEAKERS = None

MERGE_SAME_SPEAKER = True
MERGE_MAX_GAP_SECONDS = 1.25
MERGE_MAX_CHARS = 1800
SKIP_EXISTING = False

SUPPORTED_EXTENSIONS = [
    ".mp3", ".wav", ".m4a", ".aac", ".flac", ".ogg", ".opus", ".wma",
    ".mp4", ".mov", ".mkv", ".webm"
]

INITIAL_PROMPT = """
Transkrip Bahasa Indonesia yang rapi dan akurat.
Pertahankan nama orang, tempat, lembaga, dan istilah penting dengan benar.
Konteks umum: berita, wawancara, liputan, pendidikan, politik daerah, Semarang, Jawa Tengah, DPRD, Komisi E, Golkar, Dipa Yustiapasa, SPMB, SMA Negeri 3 Semarang.
"""

os.makedirs(AUDIO_FOLDER, exist_ok=True)
os.makedirs(OUTPUT_FOLDER, exist_ok=True)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
COMPUTE_TYPE = "float16" if DEVICE == "cuda" else "int8"

print("✅ Config updated with smaller batch size.")
print("Device       :", DEVICE)
print("Batch Size   :", BATCH_SIZE)

✅ Config updated with smaller batch size.
Device       : cuda
Batch Size   : 4


In [6]:
# ============================================================
# CELL 4 — HUGGINGFACE TOKEN
# ============================================================
# Multi-speaker diarization butuh HuggingFace token.
# Cara paling aman di Colab:
# 1. Klik ikon kunci/secrets di sidebar Colab
# 2. Tambahkan secret bernama HF_TOKEN
# 3. Isi value dengan token HuggingFace kamu
#
# Kalau belum pakai secrets, cell ini akan minta paste token manual.

HF_TOKEN = None

try:
    from google.colab import userdata
    HF_TOKEN = userdata.get("HF_TOKEN")
except Exception:
    HF_TOKEN = None

if not HF_TOKEN:
    HF_TOKEN = input("Paste HuggingFace token kamu di sini: ").strip()

if not HF_TOKEN:
    raise ValueError("HF_TOKEN kosong. Diarization tidak bisa jalan tanpa HuggingFace token.")

login(token=HF_TOKEN)
print("✅ HuggingFace login success.")


Paste HuggingFace token kamu di sini: 
✅ HuggingFace login success.


In [13]:
# ============================================================
# CELL 5 — LOAD WHISPERX MODELS
# ============================================================

import whisperx
import os

# Ensure token is in environment for underlying libraries
if 'HF_TOKEN' in locals() and HF_TOKEN:
    os.environ["HUGGINGFACE_HUB_TOKEN"] = HF_TOKEN

print(f"Loading WhisperX ASR model: {MODEL_NAME}")

try:
    asr_model = whisperx.load_model(
        MODEL_NAME,
        DEVICE,
        compute_type=COMPUTE_TYPE,
        language=LANGUAGE,
        asr_options={
            "initial_prompt": INITIAL_PROMPT,
            "condition_on_previous_text": True
        }
    )
except TypeError:
    # Fallback untuk versi WhisperX yang tidak support asr_options
    asr_model = whisperx.load_model(
        MODEL_NAME,
        DEVICE,
        compute_type=COMPUTE_TYPE,
        language=LANGUAGE
    )

print("✅ ASR model loaded.")

print("Loading alignment model...")
align_model, align_metadata = whisperx.load_align_model(
    language_code=LANGUAGE,
    device=DEVICE
)
print("✅ Alignment model loaded.")

print("Loading diarization model...")
try:
    # FIX: use_auth_token is removed from __init__ in newer versions
    # It now uses HUGGINGFACE_HUB_TOKEN from os.environ
    diarize_model = whisperx.diarize.DiarizationPipeline(
        model_name="pyannote/speaker-diarization-3.1",
        device=DEVICE
    )
    print("✅ Diarization model loaded.")
except Exception as e:
    print(f"❌ Diarization Load Error: {e}")
    print("\nIMPORTANT: Please visit https://hf.co/pyannote/speaker-diarization-3.1 and accept terms.")

Loading WhisperX ASR model: large-v3
2026-06-17 17:43:17 - whisperx.vads.pyannote - INFO - Performing voice activity detection using Pyannote...


INFO: Lightning automatically upgraded your loaded checkpoint from v1.5.4 to v2.6.5. To apply the upgrade to your files permanently, run `python -m lightning.pytorch.utilities.upgrade_checkpoint ../usr/local/lib/python3.12/dist-packages/whisperx/assets/pytorch_model.bin`
INFO:lightning.pytorch.utilities.migration.utils:Lightning automatically upgraded your loaded checkpoint from v1.5.4 to v2.6.5. To apply the upgrade to your files permanently, run `python -m lightning.pytorch.utilities.upgrade_checkpoint ../usr/local/lib/python3.12/dist-packages/whisperx/assets/pytorch_model.bin`


✅ ASR model loaded.
Loading alignment model...
✅ Alignment model loaded.
Loading diarization model...
2026-06-17 17:43:20 - whisperx.diarize - INFO - Loading diarization model: pyannote/speaker-diarization-3.1


plda/xvec_transform.npz:   0%|          | 0.00/134k [00:00<?, ?B/s]

plda/plda.npz:   0%|          | 0.00/134k [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/26.6M [00:00<?, ?B/s]

✅ Diarization model loaded.


In [14]:
# ============================================================
# CELL 6 — HELPER FUNCTIONS
# ============================================================

def format_timestamp_comma(seconds: float) -> str:
    """Format: 00:00:00,100"""
    if seconds is None:
        seconds = 0.0

    total_ms = int(round(float(seconds) * 1000))
    hours = total_ms // 3_600_000
    minutes = (total_ms % 3_600_000) // 60_000
    secs = (total_ms % 60_000) // 1000
    millis = total_ms % 1000

    return f"{hours:02d}:{minutes:02d}:{secs:02d},{millis:03d}"


def format_timestamp_dot(seconds: float) -> str:
    """Format VTT: 00:00:00.100"""
    return format_timestamp_comma(seconds).replace(",", ".")


def clean_text(text: str) -> str:
    if not text:
        return ""
    return " ".join(text.strip().split())


def normalize_speaker_label(raw_speaker):
    """
    Ubah label pyannote/WhisperX:
    SPEAKER_00 -> Speaker 0
    SPEAKER_01 -> Speaker 1
    """
    if raw_speaker is None:
        return "Speaker ?"

    raw = str(raw_speaker)

    match = re.search(r"(\d+)$", raw)
    if match:
        return f"Speaker {int(match.group(1))}"

    return raw.replace("_", " ").title()


def speaker_from_words(segment):
    """
    Kadang WhisperX hanya memberi speaker di level word.
    Fungsi ini ambil speaker paling dominan dari words.
    """
    words = segment.get("words", []) or []
    speakers = []

    for word in words:
        spk = word.get("speaker")
        if spk:
            speakers.append(spk)

    if speakers:
        return Counter(speakers).most_common(1)[0][0]

    return segment.get("speaker")


def normalize_segments(result_segments):
    """
    Ubah hasil WhisperX menjadi list segment standar:
    {
      start, end, speaker, text
    }
    """
    normalized = []

    for i, seg in enumerate(result_segments):
        text = clean_text(seg.get("text", ""))

        if not text:
            continue

        raw_speaker = seg.get("speaker") or speaker_from_words(seg)
        speaker = normalize_speaker_label(raw_speaker)

        normalized.append({
            "id": i,
            "start": float(seg.get("start", 0.0)),
            "end": float(seg.get("end", 0.0)),
            "speaker": speaker,
            "text": text
        })

    return normalized


def merge_same_speaker_segments(segments, max_gap=1.25, max_chars=1800):
    """
    Gabungkan segment berurutan dari speaker yang sama.
    Biar output lebih mirip:
    00:00:33,940 --> 00:03:27,580 [Speaker 1]
    Paragraf panjang...
    """
    if not segments:
        return []

    merged = []
    current = dict(segments[0])

    for seg in segments[1:]:
        same_speaker = seg["speaker"] == current["speaker"]
        gap = seg["start"] - current["end"]
        combined_len = len(current["text"]) + 1 + len(seg["text"])

        if same_speaker and gap <= max_gap and combined_len <= max_chars:
            current["end"] = seg["end"]
            current["text"] = clean_text(current["text"] + " " + seg["text"])
        else:
            merged.append(current)
            current = dict(seg)

    merged.append(current)

    for idx, seg in enumerate(merged):
        seg["id"] = idx

    return merged


def find_audio_files(folder_path: str):
    folder = Path(folder_path)

    files = []
    for file_path in folder.rglob("*"):
        if file_path.is_file() and file_path.suffix.lower() in SUPPORTED_EXTENSIONS:
            files.append(file_path)

    return sorted(files)


def save_txt_like_example(segments, output_path):
    with open(output_path, "w", encoding="utf-8") as f:
        for seg in segments:
            start = format_timestamp_comma(seg["start"])
            end = format_timestamp_comma(seg["end"])
            speaker = seg["speaker"]
            text = clean_text(seg["text"])

            if not text:
                continue

            f.write(f"{start} --> {end} [{speaker}]\n")
            f.write(f"{text}\n\n")


def save_markdown_transcript(segments, output_path, title="Transkrip Audio"):
    with open(output_path, "w", encoding="utf-8") as f:
        f.write(f"# {title}\n\n")

        for seg in segments:
            start = format_timestamp_comma(seg["start"])
            end = format_timestamp_comma(seg["end"])
            speaker = seg["speaker"]
            text = clean_text(seg["text"])

            if not text:
                continue

            f.write(f"### {start} → {end} [{speaker}]\n\n")
            f.write(f"{text}\n\n")


def save_srt(segments, output_path, include_speaker=True):
    with open(output_path, "w", encoding="utf-8") as f:
        index = 1

        for seg in segments:
            start = format_timestamp_comma(seg["start"])
            end = format_timestamp_comma(seg["end"])
            speaker = seg["speaker"]
            text = clean_text(seg["text"])

            if not text:
                continue

            if include_speaker:
                text = f"[{speaker}] {text}"

            f.write(f"{index}\n")
            f.write(f"{start} --> {end}\n")
            f.write(f"{text}\n\n")

            index += 1


def save_vtt(segments, output_path, include_speaker=True):
    with open(output_path, "w", encoding="utf-8") as f:
        f.write("WEBVTT\n\n")

        for seg in segments:
            start = format_timestamp_dot(seg["start"])
            end = format_timestamp_dot(seg["end"])
            speaker = seg["speaker"]
            text = clean_text(seg["text"])

            if not text:
                continue

            if include_speaker:
                text = f"[{speaker}] {text}"

            f.write(f"{start} --> {end}\n")
            f.write(f"{text}\n\n")


def save_json(segments, raw_result, output_path):
    data = {
        "model": MODEL_NAME,
        "language": LANGUAGE,
        "min_speakers": MIN_SPEAKERS,
        "max_speakers": MAX_SPEAKERS,
        "segments": segments,
        "raw_segments": raw_result.get("segments", [])
    }

    with open(output_path, "w", encoding="utf-8") as f:
        json.dump(data, f, ensure_ascii=False, indent=2)


print("✅ Helper functions ready.")


✅ Helper functions ready.


In [15]:
# ============================================================
# CELL 7 — CHECK AUDIO FILES
# ============================================================

audio_files = find_audio_files(AUDIO_FOLDER)

print(f"Total file audio/video ditemukan: {len(audio_files)}")

if len(audio_files) == 0:
    print("")
    print("⚠️ Belum ada audio/video.")
    print("Upload file ke folder:")
    print(AUDIO_FOLDER)
else:
    for i, file in enumerate(audio_files, start=1):
        print(f"{i}. {file}")


Total file audio/video ditemukan: 5
1. /content/drive/MyDrive/Whisper/AudioFiles/Salinan seg1.wav
2. /content/drive/MyDrive/Whisper/AudioFiles/Salinan seg2.wav
3. /content/drive/MyDrive/Whisper/AudioFiles/Salinan seg3.wav
4. /content/drive/MyDrive/Whisper/AudioFiles/Salinan seg4.wav
5. /content/drive/MyDrive/Whisper/AudioFiles/Salinan seg5.wav


In [19]:
# ============================================================
# CELL 8 — TRANSCRIBE + DIARIZE + EXPORT
# ============================================================

def transcribe_file_multispeaker(audio_path: Path):
    print("=" * 100)
    print(f"🎧 Processing: {audio_path.name}")

    base_name = audio_path.stem

    txt_path = os.path.join(OUTPUT_FOLDER, f"{base_name}.txt")
    md_path = os.path.join(OUTPUT_FOLDER, f"{base_name}.md")
    srt_path = os.path.join(OUTPUT_FOLDER, f"{base_name}.srt")
    vtt_path = os.path.join(OUTPUT_FOLDER, f"{base_name}.vtt")
    json_path = os.path.join(OUTPUT_FOLDER, f"{base_name}.json")

    if SKIP_EXISTING and os.path.exists(txt_path):
        print(f"⏭️ Skip karena output sudah ada: {txt_path}")
        return {
            "audio": str(audio_path),
            "txt": txt_path,
            "md": md_path,
            "srt": srt_path,
            "vtt": vtt_path,
            "json": json_path,
            "skipped": True
        }

    print("🔊 Loading audio...")
    audio = whisperx.load_audio(str(audio_path))

    print("📝 Transcribing...")
    result = asr_model.transcribe(
        audio,
        batch_size=BATCH_SIZE,
        language=LANGUAGE
    )

    print("🔗 Aligning words/timestamps...")
    result = whisperx.align(
        result["segments"],
        align_model,
        align_metadata,
        audio,
        DEVICE,
        return_char_alignments=False
    )

    print("👥 Detecting speakers / diarization...")

    diarize_kwargs = {}
    if MIN_SPEAKERS is not None:
        diarize_kwargs["min_speakers"] = MIN_SPEAKERS
    if MAX_SPEAKERS is not None:
        diarize_kwargs["max_speakers"] = MAX_SPEAKERS

    diarize_segments = diarize_model(str(audio_path), **diarize_kwargs)

    print("🧩 Assigning speakers to transcript...")
    result = whisperx.assign_word_speakers(diarize_segments, result)

    segments = normalize_segments(result["segments"])

    if MERGE_SAME_SPEAKER:
        segments = merge_same_speaker_segments(
            segments,
            max_gap=MERGE_MAX_GAP_SECONDS,
            max_chars=MERGE_MAX_CHARS
        )

    print("")
    print("Preview:")
    for seg in segments[:20]:
        print(
            f'{format_timestamp_comma(seg["start"])} --> {format_timestamp_comma(seg["end"])} '
            f'[{seg["speaker"]}] {seg["text"]}'
        )

    if len(segments) > 20:
        print(f"... {len(segments) - 20} segment lainnya")

    save_txt_like_example(segments, txt_path)
    save_markdown_transcript(segments, md_path, title=base_name)
    save_srt(segments, srt_path, include_speaker=True)
    save_vtt(segments, vtt_path, include_speaker=True)
    save_json(segments, result, json_path)

    print("")
    print("✅ Saved outputs:")
    print("TXT :", txt_path)
    print("MD  :", md_path)
    print("SRT :", srt_path)
    print("VTT :", vtt_path)
    print("JSON:", json_path)

    return {
        "audio": str(audio_path),
        "txt": txt_path,
        "md": md_path,
        "srt": srt_path,
        "vtt": vtt_path,
        "json": json_path,
        "skipped": False
    }


results = []

if len(audio_files) == 0:
    print("⚠️ Tidak ada file yang diproses.")
    print("Upload audio/video ke:")
    print(AUDIO_FOLDER)
else:
    for audio_file in audio_files:
        try:
            result = transcribe_file_multispeaker(audio_file)
            results.append(result)
        except Exception as e:
            print("❌ Error saat memproses:", audio_file)
            print(type(e).__name__, ":", e)
            print("")
            print("Tips:")
            print("- Pastikan GPU aktif.")
            print("- Pastikan HF_TOKEN benar.")
            print("- Pastikan kamu sudah accept terms pyannote di HuggingFace.")
            print("- Kalau VRAM error, turunkan BATCH_SIZE ke 8.")
            print("- Kalau speaker kebanyakan/kurang, set MIN_SPEAKERS dan MAX_SPEAKERS.")

print("=" * 100)
print("✅ Semua proses selesai.")
print(f"Total output: {len(results)} file diproses/dicek.")


🎧 Processing: Salinan seg1.wav
🔊 Loading audio...
📝 Transcribing...
❌ Error saat memproses: /content/drive/MyDrive/Whisper/AudioFiles/Salinan seg1.wav
RuntimeError : parallel_for failed: cudaErrorInvalidDevice: invalid device ordinal

Tips:
- Pastikan GPU aktif.
- Pastikan HF_TOKEN benar.
- Pastikan kamu sudah accept terms pyannote di HuggingFace.
- Kalau VRAM error, turunkan BATCH_SIZE ke 8.
- Kalau speaker kebanyakan/kurang, set MIN_SPEAKERS dan MAX_SPEAKERS.
🎧 Processing: Salinan seg2.wav
🔊 Loading audio...
📝 Transcribing...
🔗 Aligning words/timestamps...
👥 Detecting speakers / diarization...
🧩 Assigning speakers to transcript...

Preview:
00:00:16,482 --> 00:00:55,165 [Speaker 0] Ya baik, seperti yang saya janjikan tadi, saya sekarang ada di SMA 3 Semarang dan hari ini saya tuh seneng banget karena ikut merasakan aura dulu gimana sih mendaftar waktu SMA gitu ya. Dan secara tidak langsung ternyata saya merasa, wah saya muda banget saat ini. Dan sudah bersama orang-orang hebat hari i

In [20]:
# ============================================================
# CELL 9 — RESULT SUMMARY
# ============================================================

if "results" in globals() and len(results) > 0:
    print("📁 Ringkasan output:")
    for item in results:
        print("-" * 100)
        print("Audio:", item["audio"])
        print("TXT  :", item["txt"])
        print("MD   :", item["md"])
        print("SRT  :", item["srt"])
        print("VTT  :", item["vtt"])
        print("JSON :", item["json"])
else:
    print("Belum ada results. Jalankan CELL 8 dulu.")


📁 Ringkasan output:
----------------------------------------------------------------------------------------------------
Audio: /content/drive/MyDrive/Whisper/AudioFiles/Salinan seg2.wav
TXT  : /content/drive/MyDrive/Whisper/Transcripts_MultiSpeaker/Salinan seg2.txt
MD   : /content/drive/MyDrive/Whisper/Transcripts_MultiSpeaker/Salinan seg2.md
SRT  : /content/drive/MyDrive/Whisper/Transcripts_MultiSpeaker/Salinan seg2.srt
VTT  : /content/drive/MyDrive/Whisper/Transcripts_MultiSpeaker/Salinan seg2.vtt
JSON : /content/drive/MyDrive/Whisper/Transcripts_MultiSpeaker/Salinan seg2.json
----------------------------------------------------------------------------------------------------
Audio: /content/drive/MyDrive/Whisper/AudioFiles/Salinan seg3.wav
TXT  : /content/drive/MyDrive/Whisper/Transcripts_MultiSpeaker/Salinan seg3.txt
MD   : /content/drive/MyDrive/Whisper/Transcripts_MultiSpeaker/Salinan seg3.md
SRT  : /content/drive/MyDrive/Whisper/Transcripts_MultiSpeaker/Salinan seg3.srt
VTT  : 

## Output `.txt`

Output akan seperti ini:

```txt
00:00:00,100 --> 00:00:33,940 [Speaker 0]
[musik intro bersemangat]

00:00:33,940 --> 00:03:27,580 [Speaker 1]
Isi transkrip pembicara pertama...

00:03:27,800 --> 00:03:34,500 [Speaker 2]
Isi transkrip pembicara kedua...
```

## Kalau speaker masih ngawur

Diarization itu mendeteksi **karakter suara**, bukan nama orang. Jadi hasilnya tetap `[Speaker 0]`, `[Speaker 1]`, bukan otomatis “Dipa”, “Host”, dan sebagainya.

Kalau speaker terlalu banyak atau terlalu sedikit, set manual di CELL 3:

```python
MIN_SPEAKERS = 2
MAX_SPEAKERS = 3
```

Kalau audio ada musik, noise, atau orang saling potong, diarization bisa kurang akurat.
